# Star Schema with SCD Type 2 Dimensions

## Overview
This notebook transforms the flat drone radiation data into a normalized star schema with Slowly Changing Dimension (SCD) Type 2 tracking for device calibrations.

### Data Model Components
* **Fact Table**: `fact_radiation_readings` - radiation levels measured at specific locations and times
* **Dimension Tables**:
  * `dim_location` - GPS unit information and coordinates (SCD Type 2)
  * `dim_detector` - radiation detector equipment details (3 types)
  * `dim_date` - date hierarchy for temporal analysis

## Step 1 — Read the Source

Load the drone radiation data from S3.

In [0]:
from pyspark.sql import functions as F

SOURCE = "s3a://data5035-spring26/drone_data.json"
source_df = spark.read.json(SOURCE)

display(source_df.limit(15))

## Step 2 — Build Dimension Tables

### Create `dim_location`
GPS unit information and coordinates.
* **SCD Type 2**: Tracks calibration changes over time
* **Key**: Composite of GPS unit number and calibration timestamp

In [0]:
from pyspark.sql.functions import col, to_timestamp, sha2, concat_ws

dim_location = source_df.select(
    col("GPS_UNIT_NUMBER"),
    col("GPS_LAT"),
    col("GPS_LNG"),
    col("GPS_UNIT_CALIBRATION_PRECISION"),
    to_timestamp(col("GPS_UNIT_CALIBRATION_TIMESTAMP")).alias("GPS_UNIT_CALIBRATION_TIMESTAMP")
).dropDuplicates(["GPS_UNIT_NUMBER", "GPS_UNIT_CALIBRATION_TIMESTAMP"])  # SCD Type 2: Keep all calibration versions

# Add surrogate key
dim_location = dim_location.withColumn(
    "location_key",
    sha2(concat_ws("|", col("GPS_UNIT_NUMBER"), col("GPS_UNIT_CALIBRATION_TIMESTAMP").cast("string")), 256)
)

print("=== dim_location ===")
display(dim_location.limit(10))

### Create `dim_detector`
Union all 3 detector types into one dimension table.
* **Detector Types**: CESIUM_137, GAMMA, THORIUM_232
* **Attributes**: Unit number, type, calibration precision and timestamp
* **Key**: Composite of detector unit number and type

In [0]:
from pyspark.sql.functions import lit

cesium = source_df.select(
    col("CESIUM_137_DETECTOR_UNIT_NUMBER").alias("detector_unit_number"),
    lit("CESIUM_137").alias("detector_type"),
    col("CESIUM_137_DETECTOR_CALIBRATION_PRECISION").alias("calibration_precision"),
    to_timestamp(col("CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP")).alias("calibration_timestamp")
)

gamma = source_df.select(
    col("GAMMA_DETECTOR_UNIT_NUMBER").alias("detector_unit_number"),
    lit("GAMMA").alias("detector_type"),
    col("GAMMA_DETECTOR_CALIBRATION_PRECISION").alias("calibration_precision"),
    to_timestamp(col("GAMMA_DETECTOR_CALIBRATION_TIMESTAMP")).alias("calibration_timestamp")
)

thorium = source_df.select(
    col("THORIUM_232_DETECTOR_UNIT_NUMBER").alias("detector_unit_number"),
    lit("THORIUM_232").alias("detector_type"),
    col("THORIUM_232_DETECTOR_CALIBRATION_PRECISION").alias("calibration_precision"),
    to_timestamp(col("THORIUM_232_DETECTOR_CALIBRATION_TIMESTAMP")).alias("calibration_timestamp")
)

dim_detector = cesium.union(gamma).union(thorium) \
    .dropDuplicates(["detector_unit_number", "detector_type"])

# Add surrogate key
dim_detector = dim_detector.withColumn(
    "detector_key",
    sha2(concat_ws("|", col("detector_unit_number"), col("detector_type")), 256)
)

print("=== dim_detector ===")
display(dim_detector.limit(10))

### Create `dim_date`
Date dimension derived from collection timestamps.
* **Attributes**: Date, year, month, day
* **Key**: Date as string (YYYY-MM-DD)

In [0]:
from pyspark.sql.functions import to_date, year, month, dayofmonth

dim_date = source_df.select(
    to_date(col("COLLECTION_TIMESTAMP")).alias("date_actual")
).dropDuplicates(["date_actual"]) \
 .withColumn("year",  year(col("date_actual"))) \
 .withColumn("month", month(col("date_actual"))) \
 .withColumn("day",   dayofmonth(col("date_actual"))) \
 .withColumn("date_key", col("date_actual").cast("string"))

print("=== dim_date ===")
display(dim_date.limit(10))

### Step 3 — create `fact_radiation_readings`
Central fact table with one row per collection event.
* **Foreign Keys**: Link to `dim_date`, `dim_location`, and three `dim_detector` entries (one per detector type)
* **Measures**: CESIUM_137_LEVEL, GAMMA_LEVEL, THORIUM_232_LEVEL
* **Grain**: One reading per timestamp, location, and detector combination

In [0]:

fact_radiation = source_df.select(
    to_timestamp(col("COLLECTION_TIMESTAMP")).alias("collection_timestamp"),
    to_date(col("COLLECTION_TIMESTAMP")).cast("string").alias("date_key"),

    # FK to dim_location
    sha2(concat_ws("|",
        col("GPS_UNIT_NUMBER"),
        to_timestamp(col("GPS_UNIT_CALIBRATION_TIMESTAMP")).cast("string")
    ), 256).alias("location_key"),

    # FK to dim_detector (each type)
    sha2(concat_ws("|", col("CESIUM_137_DETECTOR_UNIT_NUMBER"), lit("CESIUM_137")), 256).alias("cesium_detector_key"),
    sha2(concat_ws("|", col("GAMMA_DETECTOR_UNIT_NUMBER"),     lit("GAMMA")),       256).alias("gamma_detector_key"),
    sha2(concat_ws("|", col("THORIUM_232_DETECTOR_UNIT_NUMBER"), lit("THORIUM_232")), 256).alias("thorium_detector_key"),

    # Measurements (the actual facts)
    col("CESIUM_137_LEVEL"),
    col("GAMMA_LEVEL"),
    col("THORIUM_232_LEVEL")
)

print("=== fact_radiation_readings ===")
display(fact_radiation.limit(10))

## Step 4 — Write Tables to Delta Format

Persist all dimension and fact tables as managed Delta tables in Databricks.

In [0]:
# Write dimension tables
dim_location.write.mode("overwrite").saveAsTable("dim_location")
print("✓ Written: dim_location")

dim_detector.write.mode("overwrite").saveAsTable("dim_detector")
print("✓ Written: dim_detector")

dim_date.write.mode("overwrite").saveAsTable("dim_date")
print("✓ Written: dim_date")

# Write fact table
fact_radiation.write.mode("overwrite").saveAsTable("fact_radiation_readings")
print("✓ Written: fact_radiation_readings")

print("\n=== All tables written successfully ===")